## Regime Detection

Description:     
We detect market regimes (up, down, sideways) using a simple rule—if a short moving average is above a longer one, label = +1 (“up” regime), if below, label = -1 (“down” regime), otherwise 0 (“sideways”). By identifying these regimes, we can adapt trading strategies to different market states or selectively trade only in certain conditions.

#### 📌 Important Note:
This notebook contains **interactive charts generated using Vectorbt**.  
**GitHub does not display interactive Plotly charts**, so the graphs will not be visible here.  

✅ **To view the charts**, please **download this notebook** and run it on your local machine.  
Make sure you have **Vectorbt and its dependencies installed** to regenerate the visualizations.


In [1]:
# REGIME DETECTION + LabelEncoder approach for SHIFTED labels
# Step 1: Move to project root
%cd c:/Users/moham/OneDrive/ml_bot_trading

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import MetaTrader5 as mt5
import vectorbt as vbt
import joblib

# Our modules
from data.data_loader import get_data_mt5
from features.feature_engineering import add_all_ta_features
from models.model_training import (
    select_features_rf_reg,
    walk_forward_splits
)
from backtests.simple_backtest import simulate_trading, calculate_sharpe_ratio

# Sklearn / Models
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

###########################################################
# 1) DATA LOADING & FEATURE ENGINEERING
###########################################################
if not mt5.initialize():
    print("Failed to initialize MT5")
else:
    data = get_data_mt5(symbol="BTCUSD", timeframe=mt5.TIMEFRAME_H4, n_bars=1000)
    mt5.shutdown()

df = add_all_ta_features(data)

###########################################################
# 2) Regime Detection Labeling
###########################################################
def create_labels_regime_detection(df, short_window=20, long_window=50):
    """
    Simple regime detection:
      +1 if short MA > long MA (up)
      -1 if short MA < long MA (down)
       0 otherwise (sideways)
    """
    df_copy = df.copy()
    
    df_copy["ma_short"] = df_copy["close"].rolling(short_window).mean()
    df_copy["ma_long"]  = df_copy["close"].rolling(long_window).mean()
    
    df_copy["regime_label"] = 0
    up_mask   = df_copy["ma_short"] > df_copy["ma_long"]
    down_mask = df_copy["ma_short"] < df_copy["ma_long"]
    
    df_copy.loc[up_mask, "regime_label"]   = 1
    df_copy.loc[down_mask, "regime_label"] = -1
    
    # drop rows where MAs are NaN
    df_copy.dropna(subset=["ma_short","ma_long"], inplace=True)
    
    return df_copy

df_lbl = create_labels_regime_detection(df, short_window=20, long_window=50)

# X => all features except the label columns
X = df_lbl.drop(columns=["regime_label", "ma_short", "ma_long"])
y = df_lbl["regime_label"]  # in {-1,0,+1}

###########################################################
# SHIFT LABELS? 
# We'll SHIFT them to [0,1,2] in each fold with LabelEncoder
# so we can handle folds with only two classes, e.g. {0,2}.
###########################################################

###########################################################
# 3) WALK-FORWARD SPLITS
###########################################################
folds = walk_forward_splits(X, y, n_splits=3)
print(f"Number of folds created: {len(folds)}")

###########################################################
# 4) DEFINE CLASSIFICATION MODELS
###########################################################
models = {
    "RandomForestClassifier": RandomForestClassifier(n_estimators=100, random_state=42),
    "GradientBoostingClassifier": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42),
    "SVC": SVC(C=1.0, kernel='rbf', probability=True),
    "XGBClassifier": XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    "LGBMClassifier": LGBMClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
}

###########################################################
# 5) LOOP OVER FOLDS + SIMPLE BACKTEST (with LabelEncoder)
###########################################################
fold_results = {}

for fold_i, (X_train_fold, y_train_fold, X_test_fold, y_test_fold) in enumerate(folds, start=1):
    print(f"\n===== Fold {fold_i} =====")
    print(f"Train size: {len(X_train_fold)}, Test size: {len(X_test_fold)}")
    
    # 5.1 SHIFT labels with LabelEncoder so classes become consecutive integers
    le = LabelEncoder()
    
    # Fit on the training labels (which might be e.g. {-1,+1}, => SHIFT with le)
    # or {0,1} => SHIFT, etc. This ensures e.g. {0,2} becomes {0,1}.
    le.fit(y_train_fold)
    y_train_enc = le.transform(y_train_fold)
    
    # 5.2 Feature selection
    # We can pass the encoded y to select_features_rf_reg
    # because it just needs an estimator with .fit(X, y).
    # SHIFT isn't strictly necessary for random forest's .feature_importances_,
    # but let's be consistent.
    rf_for_fs = RandomForestClassifier(n_estimators=100, random_state=42)
    X_train_sel, selected_idx = select_features_rf_reg(
        X_train_fold, y_train_enc, estimator=rf_for_fs, max_features=20
    )
    feats = X_train_fold.columns[selected_idx]
    print(f"Selected features for Fold {fold_i}: {feats.tolist()}")

    X_test_sel = X_test_fold[feats]

    # 5.3 Scale
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_sel)
    X_test_scaled  = scaler.transform(X_test_sel)

    fold_results[fold_i] = {}

    for model_name, model in models.items():
        # 5.4 Train on the encoded training labels
        model.fit(X_train_scaled, y_train_enc)

        # 5.5 Predict encoded classes on test
        preds_enc = model.predict(X_test_scaled)
        
        # 5.6 Decode them back to the original label set (which might be e.g. [-1,0,+1])
        preds_fold = le.inverse_transform(preds_enc)

        # 5.7 Evaluate accuracy on the unencoded test labels
        acc = accuracy_score(y_test_fold, preds_fold)

        # 5.8 Convert classification => signals
        # If your original labels are: +1 => up, -1 => down, 0 => sideways,
        # then we interpret them as: +1 => buy, -1 => sell, 0 => no trade
        signals = preds_fold

        # Align with test portion
        df_test_fold = df_lbl.loc[X_test_fold.index].copy()

        # 5.9 Simple backtest with cost
        daily_returns, total_return = simulate_trading(signals, df_test_fold, cost=0.0002)
        sr = calculate_sharpe_ratio(np.array(daily_returns))

        fold_results[fold_i][model_name] = {
            "Accuracy": acc,
            "TotalReturn": total_return,
            "Sharpe": sr
        }

###########################################################
# 6) PRINT RESULTS
###########################################################
for fold_i, model_dict in fold_results.items():
    print(f"\n=== Fold {fold_i} Results ===")
    for model_name, stats in model_dict.items():
        acc = stats["Accuracy"]
        ret = stats["TotalReturn"]
        sr = stats["Sharpe"]
        print(f"{model_name}: ACC={acc:.2f}, Return={ret:.2f}%, Sharpe={sr:.2f}")


c:\Users\moham\OneDrive\ml_bot_trading
Number of folds created: 3

===== Fold 1 =====
Train size: 237, Test size: 237
Selected features for Fold 1: ['trend_kst_sig', 'trend_macd_signal', 'momentum_ppo_signal', 'trend_kst', 'trend_visual_ichimoku_b', 'trend_trix', 'momentum_tsi', 'trend_aroon_up', 'volume_vpt', 'trend_visual_ichimoku_a', 'momentum_ppo', 'momentum_stoch_rsi_k', 'volatility_ui', 'momentum_stoch_rsi', 'trend_macd', 'momentum_ao', 'trend_kst_diff', 'trend_psar_down', 'trend_aroon_ind', 'trend_ichimoku_conv']


  File "c:\Users\moham\miniconda3\envs\ml\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\moham\miniconda3\envs\ml\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\moham\miniconda3\envs\ml\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\moham\miniconda3\envs\ml\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


[LightGBM] [Info] Number of positive: 172, number of negative: 65
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000287 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1375
[LightGBM] [Info] Number of data points in the train set: 237, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.725738 -> initscore=0.973107
[LightGBM] [Info] Start training from score 0.973107
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

In [2]:
# Code 2: Hyperparameter Tuning for a Classification Model (Regime Detection)

# Step 1: Move to project root
%cd c:/Users/moham/OneDrive/ml_bot_trading

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import MetaTrader5 as mt5
import joblib

# Sklearn / Models
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import accuracy_score, make_scorer
from sklearn.pipeline import Pipeline

# Your modules
from data.data_loader import get_data_mt5
from features.feature_engineering import add_all_ta_features

###########################################################
# 1) DATA LOADING & FEATURE ENGINEERING
###########################################################
if not mt5.initialize():
    print("Failed to initialize MT5")
else:
    data = get_data_mt5(symbol="BTCUSD", timeframe=mt5.TIMEFRAME_H4, n_bars=2000)
    mt5.shutdown()

df = add_all_ta_features(data)

###########################################################
# 2) Regime Detection Labeling
###########################################################
def create_labels_regime_detection(df, short_window=20, long_window=50):
    """
    Simple regime detection:
      +1 if short MA > long MA (up)
      -1 if short MA < long MA (down)
       0 otherwise (sideways)
    """
    df_copy = df.copy()
    
    # 1) Compute short and long MAs
    df_copy["ma_short"] = df_copy["close"].rolling(short_window).mean()
    df_copy["ma_long"]  = df_copy["close"].rolling(long_window).mean()
    
    # 2) Label each bar
    df_copy["regime_label"] = 0
    up_mask   = df_copy["ma_short"] > df_copy["ma_long"]
    down_mask = df_copy["ma_short"] < df_copy["ma_long"]
    
    df_copy.loc[up_mask, "regime_label"]   = 1
    df_copy.loc[down_mask, "regime_label"] = -1
    
    # 3) Drop rows where MAs are NaN
    df_copy.dropna(subset=["ma_short","ma_long"], inplace=True)
    
    return df_copy

# Create classification labels for regime detection
df_lbl = create_labels_regime_detection(df, short_window=20, long_window=50)

# Now we have columns: 'ma_short', 'ma_long', 'regime_label' in {-1,0,+1}
X_full = df_lbl.drop(columns=["regime_label", "ma_short", "ma_long"])
y_full = df_lbl["regime_label"]

# SHIFT LABELS from [-1,0,+1] => [0,1,2]
# so the classifier won't complain about negative labels
y_full_shifted = y_full + 1  # -1->0, 0->1, +1->2

print("Unique classes in y_full:", y_full.unique())
print("Unique classes in y_full_shifted:", y_full_shifted.unique())

# Ensure chronological order if needed
# X_full = X_full.sort_index()
# y_full_shifted = y_full_shifted.loc[X_full.index]

###########################################################
# 3) DEFINE YOUR TRAIN PORTION
###########################################################
# e.g., first 80% for hyperparam tuning
split_idx = int(len(X_full) * 0.8)
X_tune = X_full.iloc[:split_idx].copy()
y_tune_shifted = y_full_shifted.iloc[:split_idx].copy()

print(f"Tuning portion size: {len(X_tune)}")

###########################################################
# 4) TIME-BASED CV (TimeSeriesSplit)
###########################################################
tscv = TimeSeriesSplit(n_splits=3)

# We'll define a scoring for classification (accuracy)
scorer = make_scorer(accuracy_score)

###########################################################
# 5) BUILD A PIPELINE
###########################################################
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(random_state=42))
])

###########################################################
# 6) DEFINE PARAM DISTRIBUTIONS FOR RandomForestClassifier
###########################################################
param_distributions = {
    "clf__n_estimators": [100, 200, 300],
    "clf__max_depth": [None, 5, 10, 15],
    "clf__min_samples_split": [2, 5, 10],
    "clf__max_features": ["auto", "sqrt", 0.5]
}

###########################################################
# 7) SET UP RandomizedSearchCV
###########################################################
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=10,               # how many random combos
    scoring=scorer,          # 'accuracy' metric
    cv=tscv,                 # time-based folds
    random_state=42,
    n_jobs=-1,
    verbose=2
)

###########################################################
# 8) FIT ON TUNING PORTION
###########################################################
random_search.fit(X_tune, y_tune_shifted)

print("Best params:", random_search.best_params_)
print("Best score (accuracy):", random_search.best_score_)

best_estimator = random_search.best_estimator_

###########################################################
# 9) SAVE THE BEST ESTIMATOR
###########################################################
joblib.dump(best_estimator, "best_rf_pipeline.pkl")
print("Saved best estimator to 'best_rf_pipeline.pkl'")


c:\Users\moham\OneDrive\ml_bot_trading
Unique classes in y_full: [-1  1]
Unique classes in y_full_shifted: [0 2]
Tuning portion size: 1560
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params: {'clf__n_estimators': 200, 'clf__min_samples_split': 5, 'clf__max_features': 'sqrt', 'clf__max_depth': 10}
Best score (accuracy): 0.9264957264957264
Saved best estimator to 'best_rf_pipeline.pkl'


In [3]:
# THIRD CODE: Final Expanding Walk-Forward with VectorBT
# for a Regime Detection classification approach

# Step 1: Move to project root
%cd c:/Users/moham/OneDrive/ml_bot_trading

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import MetaTrader5 as mt5
import vectorbt as vbt
import joblib

# Our modules
from data.data_loader import get_data_mt5
from features.feature_engineering import add_all_ta_features

# We'll define a simple regime detection labeling function here:
def create_labels_regime_detection(df, short_window=20, long_window=50):
    """
    Simple regime detection:
      +1 if short MA > long MA (up)
      -1 if short MA < long MA (down)
       0 otherwise (sideways)
    """
    df_copy = df.copy()
    
    df_copy["ma_short"] = df_copy["close"].rolling(short_window).mean()
    df_copy["ma_long"]  = df_copy["close"].rolling(long_window).mean()
    
    df_copy["regime_label"] = 0
    up_mask   = df_copy["ma_short"] > df_copy["ma_long"]
    down_mask = df_copy["ma_short"] < df_copy["ma_long"]
    
    df_copy.loc[up_mask, "regime_label"]   = 1
    df_copy.loc[down_mask, "regime_label"] = -1
    
    # drop rows where MAs are NaN
    df_copy.dropna(subset=["ma_short","ma_long"], inplace=True)
    
    return df_copy

# Sklearn
from sklearn.metrics import accuracy_score

###########################################################
# 1) DATA LOADING & FEATURE ENGINEERING
###########################################################
if not mt5.initialize():
    print("Failed to initialize MT5")
else:
    data = get_data_mt5(symbol="BTCUSD", timeframe=mt5.TIMEFRAME_H4, n_bars=2000)
    mt5.shutdown()

# 1.1 Add technical features
df = add_all_ta_features(data)

# 1.2 Create regime detection labels
df_lbl = create_labels_regime_detection(df, short_window=20, long_window=50)

# We'll treat 'regime_label' in {-1,0,+1} as a 3-class classification
X = df_lbl.drop(columns=["regime_label", "ma_short", "ma_long"])
y = df_lbl["regime_label"]

# SHIFT from [-1,0,+1] => [0,1,2] for scikit-learn classification
y_shifted = y + 1  # -1->0, 0->1, +1->2

###########################################################
# 2) DEFINING EXPANDING SPLITS
###########################################################
def expanding_walk_forward_splits(X, y, n_splits=3):
    """
    Creates expanding walk-forward folds. For each fold i:
      - Train: [0 : fold_i]
      - Test:  [fold_i : fold_{i+1}]
    The last fold extends to the end.
    """
    n = len(X)
    fold_size = n // (n_splits + 1)
    splits = []
    
    for i in range(n_splits):
        train_end = (i+1) * fold_size
        if i == n_splits - 1:
            test_end = n
        else:
            test_end = (i+2) * fold_size
            if test_end > n:
                test_end = n
        
        X_train_fold = X.iloc[:train_end]
        y_train_fold = y.iloc[:train_end]
        
        X_test_fold = X.iloc[train_end:test_end]
        y_test_fold = y.iloc[train_end:test_end]
        
        splits.append((X_train_fold, y_train_fold, X_test_fold, y_test_fold))
    return splits

folds = expanding_walk_forward_splits(X, y_shifted, n_splits=3)
print(f"Number of folds: {len(folds)}")

###########################################################
# 3) LOAD BEST PIPELINE & RUN EXPANDING WALK-FORWARD
###########################################################
# This pipeline was presumably tuned on SHIFTED classes [0,1,2].
best_pipeline = joblib.load("best_rf_pipeline.pkl")
print("Loaded best pipeline from 'best_rf_pipeline.pkl'")

fees = 0.0002  # 0.02% transaction cost per trade

fold_results = {}

for i, (X_train_fold, y_train_fold, X_test_fold, y_test_fold) in enumerate(folds, start=1):
    print(f"\n=== Fold {i} ===")
    print(f"Train size: {len(X_train_fold)}, Test size: {len(X_test_fold)}")
    
    # 3.1 Fit the pipeline on SHIFTED labels in this fold's train portion
    best_pipeline.fit(X_train_fold, y_train_fold)
    
    # 3.2 Predict SHIFTED labels on the test portion
    preds_shifted = best_pipeline.predict(X_test_fold)
    
    # 3.3 SHIFT back to [-1,0,+1]
    preds = preds_shifted - 1
    
    # Evaluate accuracy vs. unshifted test labels
    # The test labels are SHIFTED, so let's SHIFT them back
    y_test_fold_unshifted = y_test_fold - 1
    acc = accuracy_score(y_test_fold_unshifted, preds)
    print(f"Fold {i} Accuracy={acc:.2f}")
    
    # Convert predicted classes => signals
    # +1 => buy, -1 => sell, 0 => no position
    signals = preds
    
    # Vectorbt backtest on the test portion
    df_test_fold = df_lbl.loc[X_test_fold.index].copy()
    close_prices = df_test_fold["close"]
    
    if len(signals) < len(close_prices):
        # pad signals if needed
        signals = np.append(signals, [0]*(len(close_prices)-len(signals)))
    signals_s = pd.Series(signals, index=close_prices.index)
    
    pf = vbt.Portfolio.from_signals(
        close_prices,
        entries=signals_s > 0,
        exits=signals_s < 0,
        init_cash=10000,
        freq='4H',
        fees=fees
    )
    
    total_return = pf.total_return()
    sharpe_ratio = pf.sharpe_ratio()
    
    print(f"Fold {i} Return={total_return:.2f}%, Sharpe={sharpe_ratio:.2f}")
    print("\nVectorbt stats for Fold", i)
    print(pf.stats())
    
    # Optional: Plot
    fig = pf.plot()
    fig.show()
    
    # Store results
    fold_results[i] = {
        "Accuracy": acc,
        "TotalReturn": total_return,
        "Sharpe": sharpe_ratio
    }

###########################################################
# 4) PRINT FOLD RESULTS
###########################################################
for i, stats in fold_results.items():
    acc = stats["Accuracy"]
    ret = stats["TotalReturn"]
    sr = stats["Sharpe"]
    print(f"\nFold {i} => ACC={acc:.2f}, Return={ret:.2f}%, Sharpe={sr:.2f}")


c:\Users\moham\OneDrive\ml_bot_trading
Number of folds: 3
Loaded best pipeline from 'best_rf_pipeline.pkl'

=== Fold 1 ===
Train size: 487, Test size: 487
Fold 1 Accuracy=0.80
Fold 1 Return=-0.11%, Sharpe=-1.86

Vectorbt stats for Fold 1
Start                         2024-06-22 12:00:00
End                           2024-09-11 12:00:00
Period                           81 days 04:00:00
Start Value                               10000.0
End Value                             8935.158723
Total Return [%]                       -10.648413
Benchmark Return [%]                    -11.71079
Max Gross Exposure [%]                      100.0
Total Fees Paid                         26.163376
Max Drawdown [%]                        12.048718
Max Drawdown Duration            51 days 12:00:00
Total Trades                                    7
Total Closed Trades                             7
Total Open Trades                               0
Open Trade PnL                                0.0
Win Rate [%]


=== Fold 2 ===
Train size: 974, Test size: 487
Fold 2 Accuracy=0.94
Fold 2 Return=0.30%, Sharpe=3.20

Vectorbt stats for Fold 2
Start                         2024-09-11 16:00:00
End                           2024-12-01 16:00:00
Period                           81 days 04:00:00
Start Value                               10000.0
End Value                             12972.69876
Total Return [%]                        29.726988
Benchmark Return [%]                     69.93752
Max Gross Exposure [%]                      100.0
Total Fees Paid                         27.849139
Max Drawdown [%]                        13.868133
Max Drawdown Duration            44 days 16:00:00
Total Trades                                    7
Total Closed Trades                             6
Total Open Trades                               1
Open Trade PnL                         -15.382031
Win Rate [%]                            66.666667
Best Trade [%]                          23.718047
Worst Trade [%]      


=== Fold 3 ===
Train size: 1461, Test size: 490
Fold 3 Accuracy=0.94
Fold 3 Return=-0.23%, Sharpe=-2.85

Vectorbt stats for Fold 3
Start                               2024-12-01 20:00:00
End                                 2025-02-21 08:00:00
Period                                 81 days 16:00:00
Start Value                                     10000.0
End Value                                   7718.001868
Total Return [%]                             -22.819981
Benchmark Return [%]                           0.639125
Max Gross Exposure [%]                            100.0
Total Fees Paid                               27.908332
Max Drawdown [%]                              23.234181
Max Drawdown Duration                  81 days 08:00:00
Total Trades                                          8
Total Closed Trades                                   8
Total Open Trades                                     0
Open Trade PnL                                      0.0
Win Rate [%]                


Fold 1 => ACC=0.80, Return=-0.11%, Sharpe=-1.86

Fold 2 => ACC=0.94, Return=0.30%, Sharpe=3.20

Fold 3 => ACC=0.94, Return=-0.23%, Sharpe=-2.85
